# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [ ]:
#load environment variables
%reload_ext dotenv
%dotenv 

In [32]:
#execute data_engineering notebook to create data set
import pandas as pd
import os
import sys
from glob import glob

sys.path.append(os.getenv('SRC_DIR'))

from utils.logger import get_logger
_logs = get_logger(__name__)

import random

stock_files = glob(os.path.join(os.getenv('SRC_DIR'), "data/prices_csv/stocks/*.csv"))

random.seed(42)
stock_files = random.sample(stock_files, 60)

dt_list = []
for s_file in stock_files:
    _logs.info(f"Reading file: {s_file}")
    dt = pd.read_csv(s_file).assign(
        source = os.path.basename(s_file),
        ticker = os.path.basename(s_file).replace('.csv', ''),
        Date = lambda x: pd.to_datetime(x['Date'])
    )
    dt_list.append(dt)
stock_prices = pd.concat(dt_list, axis = 0, ignore_index = True)

2025-09-28 21:46:38,781, 2945325599.py, 21, INFO, Reading file: ../../05_src/data/prices_csv/stocks\TNC.csv
2025-09-28 21:46:38,877, 2945325599.py, 21, INFO, Reading file: ../../05_src/data/prices_csv/stocks\CBB.csv
2025-09-28 21:46:38,903, 2945325599.py, 21, INFO, Reading file: ../../05_src/data/prices_csv/stocks\ALDX.csv
2025-09-28 21:46:38,913, 2945325599.py, 21, INFO, Reading file: ../../05_src/data/prices_csv/stocks\GLADD.csv
2025-09-28 21:46:38,921, 2945325599.py, 21, INFO, Reading file: ../../05_src/data/prices_csv/stocks\FIXX.csv
2025-09-28 21:46:38,931, 2945325599.py, 21, INFO, Reading file: ../../05_src/data/prices_csv/stocks\ETJ.csv
2025-09-28 21:46:38,947, 2945325599.py, 21, INFO, Reading file: ../../05_src/data/prices_csv/stocks\CMCTP.csv
2025-09-28 21:46:38,953, 2945325599.py, 21, INFO, Reading file: ../../05_src/data/prices_csv/stocks\BWG.csv
2025-09-28 21:46:38,971, 2945325599.py, 21, INFO, Reading file: ../../05_src/data/prices_csv/stocks\VIAC.csv
2025-09-28 21:46:38,9

In [33]:
stock_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239659 entries, 0 to 239658
Data columns (total 9 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   Date       239659 non-null  datetime64[ns]
 1   Open       239656 non-null  float64       
 2   High       239656 non-null  float64       
 3   Low        239656 non-null  float64       
 4   Close      239656 non-null  float64       
 5   Adj Close  239656 non-null  float64       
 6   Volume     239656 non-null  float64       
 7   source     239659 non-null  object        
 8   ticker     239659 non-null  object        
dtypes: datetime64[ns](1), float64(6), object(2)
memory usage: 16.5+ MB


In [34]:
select_tickers = stock_prices['ticker'].unique().tolist()
select_tickers

['TNC',
 'CBB',
 'ALDX',
 'GLADD',
 'FIXX',
 'ETJ',
 'CMCTP',
 'BWG',
 'VIAC',
 'REI',
 'BLPH',
 'SMG',
 'MOH',
 'AMH',
 'AMAL',
 'BPYPN',
 'ERH',
 'FAMI',
 'PFG',
 'SPXC',
 'ALL',
 'RTTR',
 'EARN',
 'ZIXI',
 'TSN',
 'WST',
 'REG',
 'MNK',
 'ESGR',
 'NGD',
 'SLRX',
 'GLW',
 'ACN',
 'CSSE',
 'WORK',
 'MOS',
 'IPWR',
 'GLUU',
 'CRMT',
 'EOLS',
 'INSU',
 'BWEN',
 'BPMX',
 'LH',
 'BRQS',
 'KALU',
 'ITCB',
 'SRE',
 'GAZ',
 'AQMS',
 'NPK',
 'QRHC',
 'CGEN',
 'LEVL',
 'BGS',
 'RIV',
 'GURE',
 'TEF',
 'SYNH',
 'KEY']

In [35]:
def get_dir_size(path='.'):
    '''Returns the total size of files contained in path.'''
    total = 0
    with os.scandir(path) as it:
        for entry in it:
            if entry.is_file():
                total += entry.stat().st_size
            elif entry.is_dir():
                total += get_dir_size(entry.path)
    return total

In [36]:
import time
import shutil

In [37]:
temp = os.getenv("TEMP_DATA")
csv_dir = os.path.join(temp, "csv")
shutil.rmtree(csv_dir, ignore_errors=True)
stock_csv = os.path.join(csv_dir, "stock_px.csv")
os.makedirs(csv_dir, exist_ok=True)

In [38]:
import dask.dataframe as dd

parquet_dir = os.path.join(temp, "parquet")
shutil.rmtree(parquet_dir, ignore_errors=True)
os.makedirs(parquet_dir, exist_ok=True)

In [39]:
px_dd = dd.from_pandas(stock_prices, npartitions = len(select_tickers))


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [40]:
PRICE_DATA = os.getenv("PRICE_DATA")
import shutil
if os.path.exists(PRICE_DATA):
    shutil.rmtree(PRICE_DATA)

In [41]:
stock_prices.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'ticker'],
      dtype='object')

In [42]:
for ticker in stock_prices['ticker'].unique():
    ticker_dt = stock_prices[stock_prices['ticker'] == ticker]
    ticker_dt = ticker_dt.assign(Year = ticker_dt.Date.dt.year)
    for yr in ticker_dt['Year'].unique():
        yr_dd = dd.from_pandas(ticker_dt[ticker_dt['Year'] == yr],2)
        yr_path = os.path.join(PRICE_DATA, ticker, f"{ticker}_{yr}")
        os.makedirs(os.path.dirname(yr_path), exist_ok=True)
        yr_dd.to_parquet(yr_path, engine = "pyarrow")
    

In [43]:
import os
from glob import glob

# Write your code below.
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
dd_px = dd.read_parquet(parquet_files).set_index("ticker")


In [44]:
dd_px

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year
npartitions=60,,,,,,,,,
ACN,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32
ALDX,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
#add lags for Close and Adj Close
dd_shift = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1 = x['Close'].shift(1),
        Adj_close_lag_1 = x['Adj Close'].shift(1)
    )
)

C:\Users\Minal\AppData\Local\Temp\ipykernel_23148\446037956.py:2: UserWarning: `meta` is not specified, inferred from partial data.
Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result

  dd_shift = dd_px.groupby('ticker', group_keys=False).apply(


In [ ]:
#calculate returns
dd_rets = dd_shift.assign(
    returns = lambda x: x['Close'] / x['Close_lag_1'] - 1
)

In [ ]:
#calculate high-low range
dd_feat = dd_rets.assign(
    hi_lo_range = lambda x: x['High'] - x['Low']
)

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [75]:
# pandas data frame output
dd_feat.compute()


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_close_lag_1,returns,hi_lo_range
ticker,,,,,,,,,,,,,
ACN,2018-01-02,153.500000,154.100006,152.779999,153.839996,148.655960,3061900.0,ACN.csv,2018,NaN,NaN,NaN,1.320007
ACN,2018-01-03,152.990005,154.990005,152.990005,154.550003,149.342056,2064200.0,ACN.csv,2018,153.839996,148.655960,0.004615,2.000000
ACN,2018-01-04,155.000000,156.860001,154.770004,156.380005,151.110382,1777000.0,ACN.csv,2018,154.550003,149.342056,0.011841,2.089996
ACN,2018-01-05,156.610001,157.720001,156.130005,157.669998,152.356918,1597600.0,ACN.csv,2018,156.380005,151.110382,0.008249,1.589996
ACN,2018-01-08,157.369995,159.009995,156.839996,158.929993,153.574463,2616900.0,ACN.csv,2018,157.669998,152.356918,0.007991,2.169998
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,2003-06-26,4.040000,4.190000,3.860000,4.000000,4.000000,515300.0,ZIXI.csv,2003,4.040000,4.040000,-0.009901,0.330000
ZIXI,2003-06-27,4.000000,4.050000,3.790000,3.850000,3.850000,162400.0,ZIXI.csv,2003,4.000000,4.000000,-0.037500,0.260000
ZIXI,2003-06-30,3.840000,4.000000,3.720000,3.770000,3.770000,119900.0,ZIXI.csv,2003,3.850000,3.850000,-0.020779,0.280000


In [ ]:
#add 10 day rolling mean of returns
dd_moving_avg = dd_feat.assign(
    rolling_mean = lambda x: x['returns'].rolling(10).mean()
)

In [ ]:
#view output
dd_moving_avg.compute()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_close_lag_1,returns,hi_lo_range,rolling_mean
ticker,,,,,,,,,,,,,,
ACN,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,2001,NaN,NaN,NaN,0.29,NaN
ACN,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,2001,15.17,11.404394,-0.010547,0.25,NaN
ACN,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,2001,15.01,11.284108,-0.000666,0.46,NaN
ACN,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,2001,15.00,11.276587,-0.009333,0.27,NaN
ACN,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,2001,14.86,11.171341,0.006057,0.30,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,2003-06-26,4.04,4.19,3.86,4.00,4.000000,515300.0,ZIXI.csv,2003,4.04,4.040000,-0.009901,0.33,-0.006616
ZIXI,2003-06-27,4.00,4.05,3.79,3.85,3.850000,162400.0,ZIXI.csv,2003,4.00,4.000000,-0.037500,0.26,-0.011064
ZIXI,2003-06-30,3.84,4.00,3.72,3.77,3.770000,119900.0,ZIXI.csv,2003,3.85,3.850000,-0.020779,0.28,-0.014528


In [ ]:
#check if moving average is showing 10th row onwards
dd_moving_avg.head(15)

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_close_lag_1,returns,hi_lo_range,rolling_mean
ticker,,,,,,,,,,,,,,
ACN,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,2001,NaN,NaN,NaN,0.29,NaN
ACN,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,2001,15.17,11.404394,-0.010547,0.25,NaN
ACN,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,2001,15.01,11.284108,-0.000666,0.46,NaN
ACN,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,2001,15.00,11.276587,-0.009333,0.27,NaN
ACN,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,2001,14.86,11.171341,0.006057,0.30,NaN
ACN,2001-07-26,14.95,14.99,14.50,14.50,10.900705,6335300.0,ACN.csv,2001,14.95,11.238999,-0.030100,0.49,NaN
ACN,2001-07-27,14.51,14.59,14.50,14.51,10.908223,3524000.0,ACN.csv,2001,14.50,10.900705,0.000690,0.09,NaN
ACN,2001-07-30,14.50,14.78,14.50,14.70,11.051059,3654300.0,ACN.csv,2001,14.51,10.908223,0.013094,0.28,NaN
ACN,2001-07-31,14.71,15.01,14.60,14.96,11.246520,1429000.0,ACN.csv,2001,14.70,11.051059,0.017687,0.41,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No, it wasnt. We executed .compute() to get the output with updated table values, and its output is a dataframe.   
Doing it in Dask would have been better as dask can partition the large data and run parallely reducing the time taken to execute the operation.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.